In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests as req
import re
import json

In [2]:
# backgrounds

resp = req.get("https://www.dndwiki.io/backgrounds")
soup = BeautifulSoup(resp.text, 'lxml')

df_bg = pd.DataFrame()

for tag in soup.find_all("div", {"role": "listitem"}):
    href = tag.find("a").get('href')
    title = tag.find("h1").getText()
    perks = tag.find_all("h2")
    ability = perks[0].getText()
    skills =  perks[1].getText()

    
    df = pd.DataFrame({
        'background': [title],
        'href': [href],
        'ability': [ability],
        'skills': [skills]
    })
    df_bg = pd.concat([df_bg, df])
    
df_bg['equipment'] = ''
df_bg['languages'] = ''
df_bg['ability_desc'] = ''
df_bg = df_bg.reset_index(drop=True)
    
for index, row in df_bg.iterrows():

    resp = req.get(f"https://dndwiki.io{row.href}")
    soup = BeautifulSoup(resp.text, 'lxml')
    
    small_dict = {}

    for div in soup.find_all("div", {"class":"entry-metadata"}):
        tag = div.find("h2", {"class":"entry-metadata-label"}).getText()
        value = div.find("h2", {"class":"entry-metalabel-content"}).getText()

        small_dict[tag] = value

    row.equipment = small_dict['Equipment:Â\xa0']
    row.languages = small_dict['Languages:Â\xa0']
    row.tools = small_dict['ToolÂ\xa0Proficiencies:Â\xa0']
    row.ability_desc = soup.find("div", {'class': 'entry-rich-text w-richtext'}).find('h2').findNext('p')
               
df_bg.to_excel('backgrounds_from_dndwikiio.xlsx')
df_bg

,background,href,ability,skills,equipment,languages,ability_desc
0,Acolyte,/backgrounds/acolyte,Shelter of the Faithful,"Insight, Religion",A holy symbol (a gift to you when you entered ...,Two of your choice,"[As an acolyte, you command the respect of tho..."
1,Anthropologist,/backgrounds/anthropologist,Adept Linguist,"Insight, Religion","A leather-bound diary, a bottle of ink, an ink...",Two of your choice,"[Before becoming an adventurer, you spent much..."
2,Archaeologist,/backgrounds/archaeologist,Historical Knowledge,"History, Survival",A wooden case containing a map to a ruin or du...,One of your choice,"[Prior to becoming an adventurer, you spent mo..."
3,Augen Trust (Spy),/backgrounds/augen-trust-spy,Spy Contact,"Deception, Stealth","A crowbar, a set of dark common clothes includ...",,[You have a reliable and trustworthy contact w...
4,Azorius Functionary,/backgrounds/azorius-functionary,Legal Authority,"Insight, Intimidation","An Azorius insignia, a scroll containing the t...",Two of your choice,[You have the authority to enforce the laws of...
...,...,...,...,...,...,...,...
70,Urban Bounty Hunter,/backgrounds/urban-bounty-hunter,Ear to the Ground,"Choose two from among Deception, Insight, Pers...",A set of clothes appropriate to your duties an...,,[You are in frequent contact with people in th...
71,Urchin,/backgrounds/urchin,City Secrets,"Sleight of Hand, Stealth","A small knife, a map of the city you grew up i...",,[You know the secret patterns and flow to citi...
72,Uthgardt Tribe Member,/backgrounds/uthgardt-tribe-member,Uthgardt Heritage,"Athletics, Survival","A hunting trap, a totemic token or set of tatt...",Any one of your choice,[You have an excellent knowledge of not only y...
73,Volstrucker Agent,/backgrounds/volstrucker-agent,Shadow Network,"Deception, Stealth","A set of common clothes, a black cloak with a ...",One of your choice,[You have access to the Volstrucker shadow net...


In [3]:
# races

resp = req.get("https://www.dndwiki.io/races")
soup = BeautifulSoup(resp.text, 'lxml')

df_race = pd.DataFrame()

for tag in soup.find_all("div", {"role": "listitem"}):
    href = tag.find("a").get('href')
    title = tag.find("h1").getText()
    
    df = pd.DataFrame({
        'race': [title],
        'href': [href],
#         'stats': [stats],
#         'skills': [skills]
    })

    resp = req.get(f"https://dndwiki.io{href}")
    soup = BeautifulSoup(resp.text, 'lxml')

    for div in soup.find_all("div", {"class":"entry-metadata"}):
        tag = div.find("h2", {"class":"entry-metadata-label"}).getText()
        tag = tag.replace('Â\xa0', '')
        tag = tag.replace(':', '').replace('.', '').strip().replace(' ', '_').lower()
        value = div.find("h2", {"class":"entry-metalabel-content"}).getText()
        
        df[tag] = value

    specipics = ''
    for p in soup.find_all("p"):
        try:
            tag = p.find("strong").getText()
            value = p.getText()
            
            if tag in ['Ability Scores:', 'Size:', 'Speed:', 'Age.', 'Alignment.', 'Size.', 
                        'Flight.', 'Darkvision.', 'Swim Speed.']:
                tag = tag.replace(':', '').replace('.', '').strip().replace(' ', '_').lower()
                df[tag] = value
            elif tag in ['Language.', 'Languages.']:
                df['language'] = value
            else:
                specipics = specipics + value
                
        except AttributeError:
            pass
    df['abilities'] = specipics
        
    df_race = pd.concat([df_race, df])

df_race

,race,href,ability_scores,size,speed,age,alignment,flight,language,abilities,darkvision,swim_speed
0,Aarakocra,/races/aarakocra,Dex +2; Wis +1,Size. Aarakocra are about 5 feet tall. They ha...,"25 feet, fly 50 feet",Age. Aarakocra reach maturity by age 3. Compar...,Alignment. Most aarakocra are good and rarely ...,Flight. You have a flying speed of 50 feet. To...,"Language. You can speak, read, and write Commo...",Talons. You are proficient with your unarmed s...,NaN,NaN
0,Aasimar,/races/aasimar,Cha +2,Size. Aasimar are built like well-proportioned...,30 feet,Age. Aasimar mature at the same rate as humans...,"Alignment. Due to their celestial heritage, aa...",NaN,"Language. You can speak, read, and write Commo...",Celestial Resistance. You have resistance to n...,"Darkvision. Thanks to your celestial heritage,...",NaN
0,Bugbear,/races/bugbear,Str +2; Dex +1,Size. Bugbears are between 6 and 8 feet tall a...,30 feet,Age. Bugbears reach adulthood at age 16 and li...,Alignment. Bugbears endure a harsh existence t...,NaN,"Languages. You can speak, read, and write Comm...",Long-Limbed. When you make a melee attack on y...,Darkvision. You can see in dim light within 60...,NaN
0,Centaur,/races/centaur,Str +2; Wis +1,Size. Centaurs stand between 6 and 7 feet tall...,40 feet,Age. Centaurs mature and age at about the same...,Alignment. Centaurs are inclined toward neutra...,NaN,"Languages. You can speak, read, and write Comm...","Fey. Your creature type is fey, rather than hu...",NaN,NaN
0,Changeling,/races/changeling,Cha +2; Choose any +1,Size. Your size is Medium.,30 feet,Age. Changelings mature slightly faster than h...,Alignment. Changelings tend toward pragmatic n...,NaN,"Languages. You can speak, read, and write Comm...","Shapechanger. As an action, you can change you...",NaN,NaN
0,Dragonborn,/races/dragonborn,Str +2; Cha +1,Size. Dragonborn are taller and heavier than h...,30 feet,Age. Young dragonborn grow quickly. They walk ...,"Alignment. Dragonborn tend to extremes, making...",NaN,"Languages. You can speak, read, and write Comm...",Draconic Ancestry. You have draconic ancestry....,NaN,NaN
0,Dwarf,/races/dwarf,Con +2,Size. Dwarves stand between 4 and 5 feet tall ...,25 feet,Age. Dwarves mature at the same rate as humans...,"Alignment. Most dwarves are lawful, believing ...",NaN,NaN,Speed. Your speed is not reduced by wearing he...,NaN,NaN
0,Elf,/races/elf,Dex +2,Size. Elves range from under 5 to over 6 feet ...,30 feet,Age. Although elves reach physical maturity at...,"Alignment. Elves love freedom, variety, and se...",NaN,"Languages. You can speak, read, and write Comm...",Keen Senses. You have proficiency in the Perce...,Darkvision. Accustomed to twilit forests and t...,NaN
0,Firbolg,/races/firbolg,Wis +2; Str +1,Size. Firbolg are between 7 and 8 feet tall an...,30 feet,"Age. As humanoids related to the fey, firbolg ...",Alignment. As people who follow the rhythm of ...,NaN,"Languages. You can speak, read, and write Comm...",Firbolg Magic. You can cast detect magic and d...,NaN,NaN
0,Genasi,/races/genasi,Con +2,Size. Genasi are as varied as their mortal par...,30 feet,Age. Genasi mature at about the same rate as h...,"Alignment. Independent and self-reliant, genas...",NaN,"Languages. You can speak, read, and write Comm...",,NaN,NaN


In [4]:
df_race.to_excel('races_from_dndwikiio.xlsx')

<h1>Classes</h1>

In [5]:
resp = req.get("https://www.dndwiki.io")
soup = BeautifulSoup(resp.text, 'lxml')

df_classes = pd.DataFrame()
df_subclasses = pd.DataFrame()

for tag in soup.find_all("div", {"class": "list-menu-item-container"}):
    
    href = tag.find("a").get('href')
    title = tag.find("a").getText()

    
    df = pd.DataFrame({
        'class_en': [title],
        'href': [href]
    })
    df_classes = pd.concat([df_classes, df])
    
    subclasses = tag.find_all("div", {"class": "collection-item-2 w-dyn-item"})
    
    for scl in subclasses:
        subclass_title = scl.find("a").getText()
        subclass_href = scl.find("a").get('href')
        
        df = pd.DataFrame({
            'class_en': [title],
            'subclass': subclass_title,
            'href': subclass_href
        })
    
        df_subclasses = pd.concat([df_subclasses, df])
    
# remove UA cause duuuh
# df_classes = df_classes[df_classes['class'].str.contains('UA')==False]

df_classes = df_classes.reset_index(drop=True)
df_subclasses = df_subclasses.reset_index(drop=True)
df_classes

,class_en,href
0,Artificer,https://www.dndwiki.io/classes/artificer
1,Barbarian,https://www.dndwiki.io/classes/barbarian
2,Bard,https://www.dndwiki.io/classes/bard
3,Cleric,https://www.dndwiki.io/classes/cleric
4,Druid,https://www.dndwiki.io/classes/druid
5,Fighter,https://www.dndwiki.io/classes/fighter
6,Monk,https://www.dndwiki.io/classes/monk
7,Mystic (UA),https://www.dndwiki.io/classes/mystic
8,Paladin,https://www.dndwiki.io/classes/paladin
9,Ranger,https://www.dndwiki.io/classes/ranger


In [6]:
df_classes['table'] = ''
for index, row in df_classes.iterrows():
    href = df_classes['href'][index]
    resp = req.get(href)

    soup = BeautifulSoup(resp.text, 'lxml')
    table = soup.find("table")
    table = pd.read_html(str(table))[0]
    json_table = table.to_json()
    
    df_classes['table'][index] = json_table
    
df_classes

,class_en,href,table
0,Artificer,https://www.dndwiki.io/classes/artificer,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
1,Barbarian,https://www.dndwiki.io/classes/barbarian,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
2,Bard,https://www.dndwiki.io/classes/bard,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
3,Cleric,https://www.dndwiki.io/classes/cleric,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
4,Druid,https://www.dndwiki.io/classes/druid,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
5,Fighter,https://www.dndwiki.io/classes/fighter,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
6,Monk,https://www.dndwiki.io/classes/monk,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
7,Mystic (UA),https://www.dndwiki.io/classes/mystic,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
8,Paladin,https://www.dndwiki.io/classes/paladin,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."
9,Ranger,https://www.dndwiki.io/classes/ranger,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4..."


In [7]:
df_classes['descriptions'] = ''
for index, row in df_classes.iterrows():
    href = df_classes['href'][index]
    resp = req.get(href)

    soup = BeautifulSoup(resp.text, 'lxml')
    
    all_tags = soup.find_all()
    attr = 'start'
    attr_values = []
    attr_dict = {}

    for tag in all_tags:
        if 'h2' in tag.name or 'h3' in tag.name:
            if tag.attrs:
                pass
            else:
                attr_dict[attr] = attr_values
                attr = tag.getText()
                attr_values = []
        if "p" in tag.name:
            attr_value = tag.getText()
            attr_values.append(attr_value)

    attr_dict['start'] = []
    attr_dict = json.dumps(attr_dict)
    df_classes['descriptions'][index] = attr_dict

In [8]:
df_subclasses['table'] = ''
for index, row in df_subclasses.iterrows():
    href = df_subclasses['href'][index]
    resp = req.get("https://www.dndwiki.io/"+href)

    soup = BeautifulSoup(resp.text, 'lxml')
    table = soup.find("table")
    try:
        table = pd.read_html(str(table))[0]
        json_table = table.to_json()
    except:
        json_table = ''
    df_subclasses['table'][index] = json_table
    
df_subclasses

,class_en,subclass,href,table
0,Artificer,Alchemist,/subclasses/alchemist,
1,Artificer,Armorer (UA),/subclasses/armorer-ua,
2,Artificer,Artillerist,/subclasses/artillerist,
3,Artificer,Battle Smith,/subclasses/battle-smith,
4,Barbarian,Path of the Ancestral Guardian,/subclasses/path-of-the-ancestral-guardian,
...,...,...,...,...
177,Wizard,School of Necromancy,/subclasses/school-of-necromancy,
178,Wizard,School of Transmutation,/subclasses/school-of-transmutation,
179,Wizard,Technomancy (UA),/subclasses/technomancy,
180,Wizard,Theurgy (UA),/subclasses/theurgy,


In [9]:
df_subclasses['descriptions'] = ''
for index, row in df_subclasses.iterrows():
    href = df_subclasses['href'][index]
    resp = req.get("https://www.dndwiki.io/"+href)

    soup = BeautifulSoup(resp.text, 'lxml')
    
    all_tags = soup.find_all()
    attr = 'start'
    attr_values = []
    attr_dict = {}

    for tag in all_tags:
        if 'h2' in tag.name or 'h3' in tag.name:
            if tag.attrs:
                pass
            else:
                attr_dict[attr] = attr_values
                attr = tag.getText()
                attr_values = []
        if "p" in tag.name:
            attr_value = tag.getText()
            attr_values.append(attr_value)

    attr_dict['start'] = []
    attr_dict = json.dumps(attr_dict)
    df_subclasses['descriptions'][index] = attr_dict
df_subclasses

,class_en,subclass,href,table,descriptions
0,Artificer,Alchemist,/subclasses/alchemist,,"{""start"": [], ""Tool Proficiency"": [""When you a..."
1,Artificer,Armorer (UA),/subclasses/armorer-ua,,"{""start"": [], ""Tools of the Trade"": [""3rd-leve..."
2,Artificer,Artillerist,/subclasses/artillerist,,"{""start"": [], ""Tool Proficiency"": [""When you a..."
3,Artificer,Battle Smith,/subclasses/battle-smith,,"{""start"": [], ""Tool Proficiency"": [""When you a..."
4,Barbarian,Path of the Ancestral Guardian,/subclasses/path-of-the-ancestral-guardian,,"{""start"": [], ""Ancestral Protectors"": [""Starti..."
...,...,...,...,...,...
177,Wizard,School of Necromancy,/subclasses/school-of-necromancy,,"{""start"": [], ""Necromancy Savant"": [""Beginning..."
178,Wizard,School of Transmutation,/subclasses/school-of-transmutation,,"{""start"": [], ""Transmutation Savant"": [""Beginn..."
179,Wizard,Technomancy (UA),/subclasses/technomancy,,"{""start"": [], ""Bonus Proficiencies"": [""Beginni..."
180,Wizard,Theurgy (UA),/subclasses/theurgy,,"{""start"": [], ""Divine Inspiration"": [""When you..."


In [10]:
df_classes.to_excel('classes_from_dndwiki.xlsx')

In [11]:
df_classes['dice'] = ''
df_classes['tools'] = ''
df_classes['proficiencies'] = ''
df_classes['starting_skills'] = ''
df_classes['saves'] = ''
df_classes['equipment'] = ''

for index, row in df_classes.iterrows():

    s = json.loads(df_classes['descriptions'][index])

    dice = s["Hit Points"][0].split("Hit Dice: ")[1]

    profs_dict = {}
    profs =  s["Proficiencies"]
    for prof in profs:
        prof = prof.split(": ")
        profs_dict[prof[0]] = prof[1]

    tools = profs_dict['Armor'] + profs_dict['Weapons'] + profs_dict['Tools']
    proficiencies = profs_dict['Skills']
    saves = profs_dict['Saving Throws']
    table = json.loads(df_classes['table'][index])


    df_classes['dice'][index] = dice
    df_classes['tools'][index] = tools
    df_classes['proficiencies'][index] = proficiencies
    df_classes['saves'][index] = saves
    if 'Features' in table:
        df_classes['starting_skills'][index] = table['Features']["0"]
    if "Starting Equipment" in s:
        df_classes['equipment'][index] = s["Starting Equipment"][0]

    
df_classes

,class_en,href,table,descriptions,dice,tools,proficiencies,starting_skills,saves,equipment
0,Artificer,https://www.dndwiki.io/classes/artificer,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,"light armor, medium armor, shieldssimple weapo...","Choose 2 from Arcana, History, Investigation, ...","Optional Rule: Firearm Proficiency, Magical Ti...","Constitution, Intelligence","You start with the following items, plus anyth..."
1,Barbarian,https://www.dndwiki.io/classes/barbarian,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d12"",...",1d12,"light armor, medium armor, shieldssimple weapo...","Choose 2 from Animal Handling, Athletics, Inti...","Rage, Unarmored Defense","Strength, Constitution","You start with the following items, plus anyth..."
2,Bard,https://www.dndwiki.io/classes/bard,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,"light armorsimple weapons, hand crossbows, lon...",Choose any 3.,"Bardic Inspiration, Magical Inspiration, Spell...","Dexterity, Charisma","You start with the following items, plus anyth..."
3,Cleric,https://www.dndwiki.io/classes/cleric,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,"light armor, medium armor, shieldssimple weapo...","Choose 2 from History, Insight, Medicine, Pers...","Spellcasting, Cantrip Versatility, Divine Domain","Wisdom, Charisma","You start with the following items, plus anyth..."
4,Druid,https://www.dndwiki.io/classes/druid,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,"light armor, medium armor, shields (druids wil...","Choose 2 from Arcana, Animal Handling, Insight...","Druidic, Spellcasting, Cantrip Versatility","Intelligence, Wisdom","You start with the following items, plus anyth..."
5,Fighter,https://www.dndwiki.io/classes/fighter,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d10"",...",1d10,"light armor, medium armor, heavy armor, shield...","Choose 2 from Acrobatics, Animal Handling, Ath...","Fighting Style, Martial Versatility, Second Wi...","Strength, Constitution","You start with the following items, plus anyth..."
6,Monk,https://www.dndwiki.io/classes/monk,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,"nonesimple weapons, shortswordsany one type of...","Choose 2 from Acrobatics, Athletics, History, ...","Unarmored Defense, Martial Arts, Monk Weapons","Strength, Dexterity","You start with the following items, plus anyth..."
7,Mystic (UA),https://www.dndwiki.io/classes/mystic,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d8"", ...",1d8,light armorsimple weaponsNone,"Choose 2 from Arcana, History, Insight, Medici...","Psionics, Psionic Disciplines and Talents, Usi...","Intelligence, Wisdom","You start with the following items, plus anyth..."
8,Paladin,https://www.dndwiki.io/classes/paladin,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d10"",...",1d10,"light armor, medium armor, heavy armor, shield...","Choose 2 from Athletics, Insight, Intimidation...","Divine Sense, Lay on Hands","Wisdom, Charisma","You start with the following items, plus anyth..."
9,Ranger,https://www.dndwiki.io/classes/ranger,"{""Level"":{""0"":""1st"",""1"":""2nd"",""2"":""3rd"",""3"":""4...","{""start"": [], ""Hit Points"": [""Hit Dice: 1d10"",...",1d10,"light armor, medium armor, shieldssimple weapo...","Choose 3 from Animal Handling, Athletics, Insi...","Favored Enemy, Favored Foe, Natural Explorer, ...",

In [12]:
df_subclasses.to_excel('subclasses_from_dndwiki.xlsx')

In [13]:
import pandas as pd
import sqlalchemy
from sqlalchemy import Column, BigInteger, String, Integer, Float, null, \
    DateTime, and_, or_, func, cast, Date, Sequence, ForeignKey, Boolean, \
    literal
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import scoped_session, sessionmaker, relationship

dwh_db = sqlalchemy.create_engine('sqlite:////Users/alinacepurnova/Desktop/Приложенька/aichepurnova/db.sqlite3')
DWHSession = sessionmaker(dwh_db)
dwh_session = DWHSession()
query_session = scoped_session(DWHSession)
base = declarative_base()
base.query = query_session.query_property()

In [14]:
dwh_db.execute("drop table classes")
dwh_db.execute("drop table subclasses")

In [15]:
df_classes.to_sql('classes', dwh_db)
df_subclasses.to_sql('subclasses', dwh_db)